# BML pyjs 流水线验证（bml-subpro 适配版）

**测试目录**: `bml-subpro/01`
**目标文件**: `01.mkv` (599 MB, H.264 + EAC3 + 9字幕轨 + 3字体附件)

> 每一步都是**独立**的 — 先检查文件夹里有什么，条件满足才执行，没有就跳过。
>
> 🆕 **本次适配变更**（相对于原版 pyjs）:
> - 🔧 修复 `_parse_add_response` SSH 横幅污染 → 做种不再假失败
> - 🔧 修复 `sync_to_server` 未校验不删除 R2 文件
> - 📦 所有生成步骤覆盖前自动备份到 `_backup/` 目录
> - ⏱️ 集成 `PipelineTimer` 时间轴追踪
> - ⚡ 大文件上传显示实时网速

In [ ]:
import sys
sys.path.insert(0, '../bml-subpro')

from bmlsub import (
    Pipeline, PipelineConfig,
    MediaExtractor, ExtractedTrack,
    Transcriber, Encoder, SubtitleValidator, Packager,
    TorrentCreator, Transfer, R2Uploader, RemoteSeeder, Publisher,
    PRESET_HEVC_VT_DEFAULT, PRESET_X264_SLOW, SUB_STANDARD_HD,
    # 🆕 新增工具模块
    PipelineTimer, ProgressBar, SpeedMeter,
    backup_if_exists,
    # 产物命名 & 检测
    product_path, product_torrent_path, check_products, scan_products,
    PRODUCT_FORMATS,
)
from pathlib import Path
import subprocess
import json
import fnmatch

EP_DIR = Path('.').resolve()
EP_ID = '01'
SRC = EP_DIR / f'{EP_ID}.mkv'

print(f'📁 工作目录: {EP_DIR}')
print(f'🎬 源文件: {SRC.name} ({SRC.stat().st_size/1024/1024:.0f} MB)')
print()
print('🆕 bml-subpro 适配版 — 已启用备份机制 + 时间轴 + 网速显示')


# ── 项目模板（换项目时只改这几行）────────────────
# 产物命名前缀（简体 / 繁体）
PREFIX_CHS = "[Billion Meta Lab] 不虐待我的继母与继姐 Ibitte Konai Gibo to Gishi"
PREFIX_CHT = "[Billion Meta Lab] 不虐待我的繼母與繼姐 Ibitte Konai Gibo to Gishi"

# R2 上传路径
R2_FOLDER = "不虐待我的继母与继姐"

# 服务器连接
SSH_ALIAS = "<your-server>"
REMOTE_DIR = "<remote-download-dir>"

# 生成产物路径（模板定义在 bmlsub/scan.py 的 PRODUCT_FORMATS 中）
mp4_chs  = product_path(EP_DIR, EP_ID, 'mp4_chs',  PREFIX_CHS, PREFIX_CHT)
mp4_cht  = product_path(EP_DIR, EP_ID, 'mp4_cht',  PREFIX_CHS, PREFIX_CHT)
mkv_hevc = product_path(EP_DIR, EP_ID, 'mkv_hevc', PREFIX_CHS, PREFIX_CHT)

# API 发布元数据
BGM_ID = 572613  # https://bgm.tv/subject/572613
NOTES = """![](https://img.20220701.xyz/file/1783566726499_002-mv-illustration.jpg)

欢迎关注Billion Meta Lab的[Telegram频道](https://t.me/Billion_Meta_Lab)、[Telegram群组](https://t.me/+mori_rnLfb0xZGM1)、[GitHub](https://github.com/microseventh)；我们会在TG上发布本组作品以及新番制作预告

如果你有什么事情想要联系组长，请发送邮件到billionmetalab@gmail.com与小柒联系哦"""

# 自检
print(f'📦 产物模板:')
print(f'   mp4_chs  = {mp4_chs.name}')
print(f'   mp4_cht  = {mp4_cht.name}')
print(f'   mkv_hevc = {mkv_hevc.name}')
print(f'   R2 路径  = {R2_FOLDER}/{EP_ID}')
print(f'   服务器   = {SSH_ALIAS}:{REMOTE_DIR}')
print(f'   简日 MP4: {"✅" if mp4_chs.exists() else "❌ 不存在"}')
print(f'   繁日 MP4: {"✅" if mp4_cht.exists() else "❌ 不存在"}')
print(f'   内封 MKV: {"✅" if mkv_hevc.exists() else "❌ 不存在"}')

---
## 阶段 1: 提取字幕（独立步骤）

从 MKV 中提取所有字幕轨，输出为 `{stem}_sub_{lang}_{index}.ass`

In [ ]:
extractor = MediaExtractor(EP_DIR)

# 1a. 列出字幕流
print('=== 字幕流列表 ===')
sub_list = extractor.list_subtitle_streams(SRC)
for s in sub_list:
    print(f'  [{s.index}] lang={s.language} title="{s.title}" codec={s.codec_name}')
print(f'共 {len(sub_list)} 条字幕轨道')

if not sub_list:
    print('⚠️ 无字幕轨道，跳过提取')
else:
    # 1b. 提取所有字幕轨
    print('\n=== 提取所有字幕轨 ===')
    all_subs = extractor.extract_subtitle_tracks(SRC)
    for t in all_subs:
        print(f'  ✅ {t.output_path.name} (lang={t.language})')
    print(f'\n✅ 共提取 {len(all_subs)} 条字幕')

---
## 阶段 1b: 提取音轨（独立步骤，供后续转录用）

In [ ]:
from tqdm import tqdm

# 统计音频流数量
streams = [s for s in extractor.probe_streams(SRC) if s.get("codec_type") == "audio"]
audio_count = len(streams)

print('=== 音轨提取 ===')
with tqdm(total=audio_count, desc="🎵 提取音轨", unit="track") as pbar:
    audio_tracks = extractor.extract_audio_tracks(SRC, progress=pbar)

for t in audio_tracks:
    print(f'  ✅ {t.output_path.name} (lang={t.language})')
print(f'\n✅ 共提取 {len(audio_tracks)} 条音轨')

---
## 阶段 2: AI 转录（独立步骤，可跳过）

In [ ]:
# 检查音轨是否存在
audio_files = sorted(EP_DIR.glob(f'{EP_ID}_audio_*.aac'))
if not audio_files:
    audio_files = sorted(EP_DIR.glob(f'{EP_ID}_audiotracker*.aac'))

if not audio_files:
    print('⚠️ 未找到音轨文件，跳过转录。请先运行阶段 1b')
else:
    audio_path = audio_files[0]
    print(f'🎵 音轨: {audio_path.name}')

    transcriber = Transcriber(
        model='mlx-community/whisper-large-v3-turbo',
        language='ja',
        chunk_sec=240,
        overlap_sec=5,
    )

    # 方法1: 直接转录 — large-v3-turbo
    print('\n### 方法1: 直接转录 (large-v3-turbo) ###')
    direct_result = transcriber.transcribe_direct(
        audio_path,
        model='mlx-community/whisper-large-v3-turbo',
    )
    print(f'→ 输出: {direct_result}')

    # 方法2: 分割转录 — medium-mlx
    print('\n### 方法2: 分割转录 (medium-mlx) ###')
    chunked_result = transcriber.transcribe_chunked(
        audio_path,
        model='mlx-community/whisper-medium-mlx',
    )
    print(f'→ 输出: {chunked_result}')


---
## 阶段 3: HEVC VideoToolbox 硬件编码（独立步骤）

In [ ]:
encoder = Encoder(PRESET_HEVC_VT_DEFAULT, PRESET_X264_SLOW)

if not SRC.exists():
    print(f'❌ 源文件不存在: {SRC.name}，跳过 HEVC 编码')
else:
    print('🎬 VideoToolbox HEVC 10bit 编码...')
    hevc_path = encoder.encode_hevc_vt(SRC)
    print(f'→ 输出: {hevc_path.name}')

    # 验证元数据
    issues = encoder.verify_metadata_clean(hevc_path)
    if not issues:
        print('✅ 元数据完全干净！源标签已全部清除')
    else:
        print(f'⚠️ 残留标签: {issues}')

---
## 阶段 4: 字幕校验 & ASS 头部标准化（独立步骤）

扫描当前文件夹内所有 `.ass` 文件，校验并标准化 ASS 头部。

In [ ]:
# ── 清理阶段 1 提取的原始字幕，只保留制作组字幕 ──


KEEP_PATTERNS = ('*.chs&jpn.ass', '*.cht&jpn.ass', '*.chs.ass', '*.cht.ass')

all_ass = sorted(EP_DIR.glob('*.ass'))
to_delete = [f for f in all_ass
             if not any(fnmatch.fnmatch(f.name, pat) for pat in KEEP_PATTERNS)]

print(f'当前目录 ASS 文件: {len(all_ass)} 个')
print(f'保留 ({len(KEEP_PATTERNS)} 种模式): {", ".join(KEEP_PATTERNS)}')

if not to_delete:
    print('✅ 无需清理')
else:
    print(f'\n🗑️  删除 {len(to_delete)} 个原始字幕:')
    for f in to_delete:
        print(f'    ✕ {f.name}')
        f.unlink()
    print('✅ 清理完成')

In [ ]:
validator = SubtitleValidator(SUB_STANDARD_HD)

ass_files = sorted(EP_DIR.glob('*.ass'))
print(f'找到 {len(ass_files)} 个 ASS 字幕文件\n')

if not ass_files:
    print('⚠️ 无 ASS 字幕文件，跳过校验')
else:
    for ass in ass_files:
        violations = validator.validate_ass_header(ass)
        if violations:
            print(f'📝 {ass.name}: 修正 {list(violations.keys())}')
            validator.standardize_ass(ass)
        else:
            print(f'✅ {ass.name}: 已合规')

---
## 阶段 5: x264 软编码 + ASS 硬字幕烧录（独立步骤）

**先检查**当前文件夹内是否有与纯数字视频匹配的 `{id}.chs&jpn.ass` 和 `{id}.cht&jpn.ass`。  
有就压制，没有就跳过。

In [ ]:
# 检查需要的前置文件
pure_mkv = EP_DIR / f'{EP_ID}.mkv'
chs_sub = EP_DIR / f'{EP_ID}.chs&jpn.ass'
cht_sub = EP_DIR / f'{EP_ID}.cht&jpn.ass'

print('=== 前置检查 ===')
print(f'  源视频: {pure_mkv.name} {"✅" if pure_mkv.exists() else "❌ 不存在"}')
print(f'  简日字幕: {chs_sub.name} {"✅" if chs_sub.exists() else "❌ 不存在"}')
print(f'  繁日字幕: {cht_sub.name} {"✅" if cht_sub.exists() else "❌ 不存在"}')

if not pure_mkv.exists():
    print('\n⚠️ 源视频不存在，跳过 x264 压制')
elif not chs_sub.exists() and not cht_sub.exists():
    print(f'\n⚠️ 未找到 {EP_ID}.chs&jpn.ass 或 {EP_ID}.cht&jpn.ass，跳过 x264 压制')
    print(f'   请先准备好匹配的字幕文件再运行此步骤')
else:
    # 使用库中的编码预设（无需手动拼接 ffmpeg 参数）
    from bmlsub import PRESET_X264_SLOW
    encode_params = PRESET_X264_SLOW.to_ffmpeg_video_params() \
                  + PRESET_X264_SLOW.to_ffmpeg_audio_params() \
                  + ['-map_metadata', '-1', '-fflags', '+bitexact', '-flags:v', '+bitexact']

    # 构建任务列表（只包含存在的字幕） — 产物路径由 product_path() 统一生成
    tasks = []
    if chs_sub.exists():
        out_path = product_path(EP_DIR, EP_ID, 'mp4_chs', PREFIX_CHS, PREFIX_CHT)
        tasks.append((chs_sub, out_path, '简体中文'))
    if cht_sub.exists():
        out_path = product_path(EP_DIR, EP_ID, 'mp4_cht', PREFIX_CHS, PREFIX_CHT)
        tasks.append((cht_sub, out_path, '繁體中文'))

    print(f'\n========== 开始 FFmpeg x264 压制 (EP{EP_ID}) ==========')
    for sub_path, out_path, label in tasks:
        # 🆕 备份旧 MP4（不再直接跳过）
        if out_path.exists():
            bak = backup_if_exists(out_path)
            if bak:
                print(f'📦 已备份旧 MP4 → {bak.name}')

        sub_abs = str(sub_path.absolute()).replace('\\', '/')
        cmd = [
            'ffmpeg', '-y', '-i', str(pure_mkv),
            '-vf', f"ass='{sub_abs}'",
        ] + encode_params + [str(out_path)]

        print(f'🎬 正在压制 ({label}): {out_path.name}')
        subprocess.run(cmd, check=True, timeout=3600)  # 🆕 添加超时
        print(f'✅ {out_path.name} 压制完成')

---
## 阶段 6: mkvmerge 封装（独立步骤）

**先检查**:
1. HEVC 视频 `{id}_HEVC10bit.mkv` 是否存在
2. `{id}.chs&jpn.ass` 和 `{id}.cht&jpn.ass` 是否存在
3. 当前文件夹内是否有字体文件 (`.ttf` / `.otf` / `.ttc`)

全部满足才封装为: **HEVC 视频 + 简繁字幕 + 字体附件**

In [ ]:
# ── 前置检查 ──
hevc_mkv = EP_DIR / f'{EP_ID}_HEVC10bit.mkv'
chs_sub = EP_DIR / f'{EP_ID}.chs&jpn.ass'
cht_sub = EP_DIR / f'{EP_ID}.cht&jpn.ass'

# 扫描字体
fonts = []
for ext in ('*.ttf', '*.otf', '*.ttc'):
    fonts.extend(EP_DIR.glob(ext))

print('=== 前置检查 ===')
print(f'  HEVC 视频: {hevc_mkv.name} {"✅" if hevc_mkv.exists() else "❌ 不存在"}')
print(f'  简日字幕: {chs_sub.name} {"✅" if chs_sub.exists() else "❌ 不存在"}')
print(f'  繁日字幕: {cht_sub.name} {"✅" if cht_sub.exists() else "❌ 不存在"}')
print(f'  字体文件: {len(fonts)} 个 {"[✅]" if fonts else "❌ 无字体"}')
if fonts:
    for f in fonts:
        print(f'    - {f.name}')

# ── 条件判断 ──
missing = []
if not hevc_mkv.exists():
    missing.append(f'{EP_ID}_HEVC10bit.mkv (请先运行阶段 3)')
if not chs_sub.exists():
    missing.append(f'{EP_ID}.chs&jpn.ass')
if not cht_sub.exists():
    missing.append(f'{EP_ID}.cht&jpn.ass')
if not fonts:
    missing.append('字体文件 (.ttf/.otf/.ttc)')

if missing:
    print(f'\n⚠️ 缺少以下文件，跳过 mkvmerge 封装:')
    for m in missing:
        print(f'    - {m}')
else:
    # ── 执行 mkvmerge ──
    mkv_output = product_path(EP_DIR, EP_ID, 'mkv_hevc', PREFIX_CHS, PREFIX_CHT)

    # 🆕 备份旧 MKV
    if mkv_output.exists():
        bak = backup_if_exists(mkv_output)
        if bak:
            print(f'\n📦 已备份旧 MKV → {bak.name}')

    cmd = ['mkvmerge', '-o', str(mkv_output), str(hevc_mkv)]

    # 简日字幕 (默认轨道)
    cmd += [
        '--language', '0:chi',
        '--track-name', '0:简体中文+日语',
        '--default-track', '0:yes',
        str(chs_sub),
    ]
    # 繁日字幕
    cmd += [
        '--language', '0:chi',
        '--track-name', '0:繁體中文+日语',
        '--default-track', '0:no',
        str(cht_sub),
    ]
    # 字体附件
    for font in fonts:
        ext = font.suffix.lower()
        if ext == '.ttf':
            mime = 'application/x-truetype-font'
        elif ext in ('.otf', '.ttc'):
            mime = 'application/vnd.ms-opentype'
        else:
            mime = 'application/octet-stream'
        cmd += ['--attachment-mime-type', mime, '--attach-file', str(font)]

    print(f'\n========== 开始 mkvmerge 封装 (EP{EP_ID}) ==========')
    print(f'  视频: {hevc_mkv.name}')
    print(f'  字幕: {chs_sub.name}, {cht_sub.name}')
    print(f'  字体: {len(fonts)} 个')
    print(f'  输出: {mkv_output.name}')

    subprocess.run(cmd, check=True, timeout=600)  # 🆕 添加超时
    print(f'✅ MKV 封装完成: {mkv_output.name}')

---
## 阶段 7: 生成种子（独立步骤）

**先检查**当前文件夹内是否有阶段 5（x264 硬压）和阶段 6（mkvmerge 封装）的产物。

对找到的每个视频文件，生成 `.torrent` 种子（**v1 only**，兼容动漫花园），
输出到视频文件同目录下。

In [ ]:
# ── 引用 cell-1 产物变量（由 product_path 生成）──
# mp4_chs / mp4_cht: 阶段 5 产物；mkv_hevc: 阶段 6 产物
targets = [p for p in (mp4_chs, mp4_cht, mkv_hevc) if p.exists()]

print('=== 前置检查 ===')
print(f'  简日 MP4: {mp4_chs.name} {"✅" if mp4_chs.exists() else "❌ 不存在"}')
print(f'  繁日 MP4: {mp4_cht.name} {"✅" if mp4_cht.exists() else "❌ 不存在"}')
print(f'  内封 MKV: {mkv_hevc.name} {"✅" if mkv_hevc.exists() else "❌ 不存在"}')
print(f'  可生成种子: {len(targets)} 个')

if not targets:
    print('\n⚠️ 未找到已压制/封装的视频文件，跳过种子生成')
    print('   请先运行阶段 5（x264 硬压）和/或阶段 6（mkvmerge 封装）')
else:
    creator = TorrentCreator()

    print(f'\n{"="*10} 开始生成种子 (EP{EP_ID}) {"="*10}')
    for video in targets:
        # 🆕 自动备份旧种子（TorrentCreator.create 内置了 backup_if_exists）
        # 🔧 v1_only 现已可用（原 notebook 报错是旧 pycache 导致，已清理）
        creator.create(video, v1_only=True)  # anibt.net 不支持 v2
        print()

    print('✅ 种子生成完成')

---
## 阶段 7b: 文件夹种子生成（独立步骤）

给定任意文件夹路径，将其整体打包为 **一个** `.torrent` 种子（**v1 only**，兼容动漫花园）。

与阶段 7 的区别：阶段 7 为单个视频文件生成种子，阶段 7b 为整个文件夹生成一个种子。

In [ ]:
# ── 文件夹种子生成 ──
# 修改 FOLDER 为你要打包的目录路径
FOLDER = EP_DIR  # 默认当前目录，可改为任意路径如 Path('/path/to/folder')

folder = Path(FOLDER)
if not folder.is_dir():
    print(f'❌ 不是有效目录: {folder}')
else:
    torrent_path = folder.parent / f'{folder.name}.torrent'
    
    print(f'📁 文件夹: {folder}')
    if torrent_path.exists():
        print(f'⏭️  种子已存在: {torrent_path.name}')
    else:
        creator = TorrentCreator()
        creator.create(folder, v1_only=True)  # 整个文件夹 → 一个种子
        print(f'\n✅ 文件夹种子生成完成')

---
## 阶段 9: R2 上传（独立步骤）

**先检查**当前文件夹内是否有阶段 5（x264 硬压）和阶段 6（mkvmerge 封装）的产物。

通过 Cloudflare R2 分片上传视频 + 种子文件。

> 凭证: `~/.config/bml/r2_config.json`

In [ ]:
# ── 引用 cell-1 产物变量（由 product_path 生成）──
# 收集产物 + .torrent
targets = []
for p in (mp4_chs, mp4_cht, mkv_hevc):
    if p.exists():
        targets.append(p)
        t = p.with_suffix(p.suffix + '.torrent')
        if t.exists():
            targets.append(t)

print('=== 前置检查 ===')
print(f'  简日 MP4: {mp4_chs.name} {"✅" if mp4_chs.exists() else "❌ 不存在"}')
print(f'  繁日 MP4: {mp4_cht.name} {"✅" if mp4_cht.exists() else "❌ 不存在"}')
print(f'  内封 MKV: {mkv_hevc.name} {"✅" if mkv_hevc.exists() else "❌ 不存在"}')
print(f'  待上传: {len(targets)} 个文件')

if not targets:
    print('\n⚠️ 未找到产物，跳过 R2 上传。请先运行阶段 5/6')
else:
    uploader = R2Uploader()
    r2_folder = f'{R2_FOLDER}/{EP_ID}'

    print(f'\n========== R2 上传 (EP{EP_ID}) ==========')
    print(f'Bucket: {uploader.bucket_name}')
    print(f'目标: {r2_folder}/')
    print()

    uploader.upload_files([str(t) for t in targets], remote_folder=r2_folder)
    print(f'\n✅ R2 上传完成')

---
## 阶段 10: R2 服务器拉取（独立步骤）

**先检查** R2 上是否有阶段 9 上传的文件，有就让服务器通过 rclone 拉取 → 校验 → 清理 R2。

### 服务器初始化（一次性）

#### 1. 安装 rclone
```bash
ssh <your-server>
sudo apt update && sudo apt install rclone -y
```

#### 2. 配置 R2 remote
```bash
ssh <your-server>
rclone config
```
交互步骤（按提示输入）：
- `n` → 新建 remote，输入名称 `r2`
- `Storage` → 选 Amazon S3 Compatible（输入数字或 `s3`）
- `provider` → 选 `Cloudflare`
- `env_auth` → `false`
- `access_key_id` → 从 `~/.config/bml/r2_config.json` 复制
- `secret_access_key` → 同上
- `endpoint` → `https://<account_id>.r2.cloudflarestorage.com`（account_id 同样从 r2_config.json 复制）
- 后续选项全部回车默认，最后 `y` 确认保存

验证：`rclone ls r2:bml` 应能列出文件。

#### 3. 中文文件名支持
```bash
ssh <your-server> "echo 'export LANG=en_US.utf8' >> ~/.bashrc"
```
（下次 SSH 登录 `ls` 即可正常显示中文；rclone 同步时也会自动设 `LANG=en_US.utf8`）

In [ ]:
# ── 引用 cell-1 模板变量 ──
uploader = R2Uploader()
r2_folder = f'{R2_FOLDER}/{EP_ID}'

r2_files = uploader.list_remote(r2_folder)

print('=== 前置检查 ===')
print(f'  R2 路径: {r2_folder}/')
print(f'  文件数: {len(r2_files)}')

if r2_files:
    for f in r2_files:
        print(f'    - {f}')
else:
    print('⚠️ R2 上无文件，跳过拉取。请先运行阶段 9')

if not r2_files:
    print()
else:
    # ── rclone 同步 → 校验 → 清理 ──
    print(f'\n========== 服务器拉取 ==========')
    print('🔧 注意：若未在同一 R2Uploader 实例上调用 upload_files()，')
    print('   哈希记录为空，sync_to_server 将拒绝删除 R2 文件（安全保护）')
    print()
    uploader.sync_to_server(
        ssh_alias=SSH_ALIAS,
        remote_dir=REMOTE_DIR,
        r2_prefix=r2_folder,
    )
    print(f'\n✅ 服务器拉取完成')

---
## 阶段 11: 远程做种（独立步骤）

**先检查**服务器上是否有阶段 10 拉取的 `.torrent` 种子文件。

通过 SSH + curl 把服务器上的种子添加到 qBittorrent（Docker，端口 8081），
跳过哈希校验直接做种（视频文件已在正确位置）。

> 凭证: `~/.config/bml/qb_config.json`

In [ ]:
# ── 引用 cell-1 模板变量 ──
# 通过 SSH 检查服务器上是否有 .torrent 文件
result = subprocess.run(
    ["ssh", SSH_ALIAS, f"ls '{REMOTE_DIR}'/*.torrent 2>/dev/null || echo ''"],
    capture_output=True, text=True, timeout=15,
)
torrent_files = [f for f in result.stdout.strip().split("\n") if f]

print('=== 前置检查 ===')
print(f'  服务器: {SSH_ALIAS}:{REMOTE_DIR}/')
print(f'  种子数: {len(torrent_files)}')

if torrent_files:
    for f in torrent_files:
        print(f'    - {f}')
else:
    print('⚠️ 服务器上无 .torrent 文件，跳过做种。请先运行阶段 10')

if not torrent_files:
    print()
else:
    # ── 添加种子到 qBittorrent 做种 ──
    seeder = RemoteSeeder(ssh_alias=SSH_ALIAS)

    print(f'\n========== 远程做种 ==========')
    result = seeder.add_torrents(
        torrent_files,
        save_path=REMOTE_DIR,
        skip_checking=True,
        paused=False,
    )

    ok_count = sum(1 for v in result.values() if v)
    print(f'\n{"✅ 全部做种成功" if ok_count == len(result) else f"⚠️ {ok_count}/{len(result)} 成功"}')
    print('🔧 预期结果: 3/3 成功（原 notebook 显示 0/3 是因为 SSH 横幅污染）')

---
## 阶段 12: API 发布（独立步骤）

**先检查**当前文件夹内是否有阶段 7 生成的 `.torrent` 种子文件。

将种子发布到 anibt.net。从 `.torrent` 文件中自动提取 info_hash、文件大小、
tracker 列表等信息，构造 magnet URI 并以 JSON 方式提交。

需配置每部番的 `bgmId`、`episodeKey`、`languages`、`notes` 等元数据。

> 凭证: `~/.config/bml/anibt_config.json`

In [ ]:
# ── 引用 cell-1 产物变量，派生种子路径 ──
# product_torrent_path 纯构造路径，不检查存在性
from bmlsub import Publisher

mp4_chs_t  = product_torrent_path(mp4_chs)
mp4_cht_t  = product_torrent_path(mp4_cht)
mkv_hevc_t = product_torrent_path(mkv_hevc)

# 各格式对应的语言标签（key 为种子路径，始终非 None）
FORMAT_META = {
    mp4_chs_t:  ("1080p", ["CHS", "JP"], "MP4",  "EMBEDDED"),
    mp4_cht_t:  ("1080p", ["CHT", "JP"], "MP4",  "EMBEDDED"),
    mkv_hevc_t: ("1080p", ["CHS", "CHT", "JP"], "MKV", "INTERNAL"),
}

torrents = [t for t in (mp4_chs_t, mp4_cht_t, mkv_hevc_t) if t.exists()]

print('=== 前置检查 ===')
print(f'  简日种子: {mp4_chs_t.name} {"✅" if mp4_chs_t.exists() else "❌ 不存在"}')
print(f'  繁日种子: {mp4_cht_t.name} {"✅" if mp4_cht_t.exists() else "❌ 不存在"}')
print(f'  内封种子: {mkv_hevc_t.name} {"✅" if mkv_hevc_t.exists() else "❌ 不存在"}')
print(f'  bgmId: {BGM_ID}')
print(f'  可发布: {len(torrents)} 个')

if not torrents:
    print('\n⚠️ 未找到种子文件，跳过发布。请先运行阶段 7')
else:
    print(f'\n========== API 发布 (EP{EP_ID}) ==========')
    for t in torrents:
        title = t.stem  # 去掉 .torrent → 视频文件名即标题
        resolution, languages, fmt, subtitle = FORMAT_META[t]
        
        print(f'\n📤 {t.name}')
        try:
            result = Publisher.publish_anibt(
                bgm_id=BGM_ID,
                title=title,
                episode_key=EP_ID,
                torrent_path=t,
                resolution=resolution,
                languages=languages,
                subtitle=subtitle,
                fmt=fmt,
                notes=NOTES,
                use_torrent_file=True,  # 直接上传种子文件（动漫花园兼容）
            )
            print(f'   ✅ 已发布: {result.get("id", "?")}')
        except Exception as e:
            print(f'   ❌ 失败: {e}')
    
    print(f'\n✅ API 发布完成')

---
## 流程总览

| 阶段 | 说明 | 前置条件 |
|------|------|----------|
| 1 提取字幕 | 从 MKV 提取所有字幕轨 | `{id}.mkv` 存在 |
| 1b 提取音轨 | 从 MKV 提取所有音轨 | `{id}.mkv` 存在 |
| 2 AI 转录 | Whisper 日文语音转文字 | 音轨 AAC 存在 |
| 3 HEVC 编码 | VideoToolbox 硬件 HEVC 10bit | `{id}.mkv` 存在 |
| 4 字幕校验 | ASS 头部标准化 | `.ass` 文件存在 |
| 5 x264 硬压 | libx264 + ASS 烧录 → MP4 | `{id}.mkv` + `{id}.chs&jpn.ass` / `{id}.cht&jpn.ass` |
| 6 MKV 封装 | mkvmerge: HEVC + 字幕 + 字体 | `{id}_HEVC10bit.mkv` + `{id}.chs&jpn.ass` + `{id}.cht&jpn.ass` + 字体 |
| 7 生成种子 | libtorrent v1 only .torrent | 阶段 5 或 阶段 6 产物 |
| 8 croc 传输 | croc P2P + SSH 直传服务器 | 阶段 5/6 产物 + SSH 密钥 |
| 9 R2 上传 | CF R2 分片上传视频 + 种子 | 阶段 5/6 产物 + ~/.config/bml/r2_config.json |
| 10 R2 拉取 | 服务器 rclone 从 R2 同步 → 校验 → 清理 | 阶段 9 + 服务器装好 rclone |
| 11 远程做种 | SSH + qB API 添加种子做种 | 阶段 10 + ~/.config/bml/qb_config.json |
| 12 API 发布 | anibt.net 种子文件上传 | 阶段 7 + ~/.config/bml/anibt_config.json |